In [1]:
import time
from datetime import datetime

import pandas as pd
import requests

pd.set_option('display.max_columns', None)


## Data source

EA's ratings page (`/games/ea-sports-fc/ratings`) is a Next.js app whose hashed
CSS classes (`Table_row__4INyY`, `Table_profileContent__Lna_E`, ...) change on
every site build — that is why the old BeautifulSoup scraper now returns **0
rows**. It also required one extra HTTP request *per player* to read the detail
page, which was slow and brittle.

Instead, the page is hydrated from a public JSON API that already contains every
stat in a single response:

```
GET https://drop-api.ea.com/rating/ea-sports-fc?locale=en&gender={0|1}&limit=100&offset=N
```

- `gender=0` → Men's football (~15.9k players), `gender=1` → Women's (~1.5k)
- `limit` caps at 100; paginate with `offset` up to `totalItems`
- each item already includes the full `stats` block, position, team, nation,
  play styles, etc. — no per-player request needed

The cells below page through this API and write the same CSV schema the old
scraper produced.


In [2]:
API_URL = 'https://drop-api.ea.com/rating/ea-sports-fc'
HEADERS = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'application/json',
    'Referer': 'https://www.ea.com/games/ea-sports-fc/ratings',
}
PAGE_SIZE = 100  # API max

# API stat key -> friendly column name (keeps the legacy CSV schema intact)
STAT_RENAME = {
    'pac': 'PAC', 'sho': 'SHO', 'pas': 'PAS', 'dri': 'DRI', 'def': 'DEF', 'phy': 'PHY',
    'acceleration': 'Acceleration', 'sprintSpeed': 'Sprint Speed',
    'positioning': 'Positioning', 'finishing': 'Finishing', 'shotPower': 'Shot Power',
    'longShots': 'Long Shots', 'volleys': 'Volleys', 'penalties': 'Penalties',
    'vision': 'Vision', 'crossing': 'Crossing', 'freeKickAccuracy': 'Free Kick Accuracy',
    'shortPassing': 'Short Passing', 'longPassing': 'Long Passing', 'curve': 'Curve',
    'dribbling': 'Dribbling', 'agility': 'Agility', 'balance': 'Balance',
    'reactions': 'Reactions', 'ballControl': 'Ball Control', 'composure': 'Composure',
    'interceptions': 'Interceptions', 'headingAccuracy': 'Heading Accuracy',
    'defensiveAwareness': 'Def Awareness', 'standingTackle': 'Standing Tackle',
    'slidingTackle': 'Sliding Tackle', 'jumping': 'Jumping', 'stamina': 'Stamina',
    'strength': 'Strength', 'aggression': 'Aggression', 'gkDiving': 'GK Diving',
    'gkHandling': 'GK Handling', 'gkKicking': 'GK Kicking',
    'gkPositioning': 'GK Positioning', 'gkReflexes': 'GK Reflexes',
}
FOOT = {1: 'Right', 2: 'Left'}


def _age(birthdate):
    """'12/20/1998 12:00:00 AM' -> age in whole years (or None)."""
    if not birthdate:
        return None
    dob = datetime.strptime(birthdate.split(' ')[0], '%m/%d/%Y')
    today = datetime.now()
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))


def _flatten(p):
    """Turn one API player object into a flat row dict."""
    name = p.get('commonName') or ' '.join(
        filter(None, [p.get('firstName'), p.get('lastName')])
    )
    row = {
        'Rank': p.get('rank'),
        'Name': name,
        'OVR': p.get('overallRating'),
        'Position': (p.get('position') or {}).get('shortLabel'),
        'Alternative positions': ', '.join(
            ap['shortLabel'] for ap in p.get('alternatePositions') or []
        ),
        'Nation': (p.get('nationality') or {}).get('label'),
        'League': p.get('leagueName'),
        'Team': (p.get('team') or {}).get('label'),
        'Age': _age(p.get('birthdate')),
        'Height': p.get('height'),
        'Weight': p.get('weight'),
        'Preferred foot': FOOT.get(p.get('preferredFoot')),
        'Weak foot': p.get('weakFootAbility'),
        'Skill moves': p.get('skillMoves'),
        'play style': ', '.join(a['label'] for a in p.get('playerAbilities') or []),
        'Gender': (p.get('gender') or {}).get('label'),
        'url': f"https://www.ea.com/games/ea-sports-fc/ratings/player-ratings/{p.get('id')}",
    }
    stats = p.get('stats') or {}
    for api_key, col in STAT_RENAME.items():
        cell = stats.get(api_key)
        row[col] = cell.get('value') if isinstance(cell, dict) else None
    return row


def fetch_players(gender, sleep=0.2):
    """Fetch every player for a gender (0=men, 1=women) via the paginated API."""
    session = requests.Session()

    def _page(offset):
        resp = session.get(
            API_URL,
            params={'locale': 'en', 'gender': gender,
                    'limit': PAGE_SIZE, 'offset': offset},
            headers=HEADERS, timeout=30,
        )
        resp.raise_for_status()
        return resp.json()

    payload = _page(0)
    total = payload['totalItems']
    rows = [_flatten(p) for p in payload['items']]

    for offset in range(PAGE_SIZE, total, PAGE_SIZE):
        rows.extend(_flatten(p) for p in _page(offset)['items'])
        print(f'  gender={gender}: {len(rows)}/{total}', end='\r')
        time.sleep(sleep)

    print(f'  gender={gender}: {len(rows)}/{total} done')
    return pd.DataFrame(rows)


In [3]:
# Men's football (~15.9k players, ~160 requests)
male_players = fetch_players(gender=0)
male_players.head()


  gender=0: 15905/15905 done


,Rank,Name,OVR,Position,Alternative positions,Nation,League,Team,Age,Height,Weight,Preferred foot,Weak foot,Skill moves,play style,Gender,url,PAC,SHO,PAS,DRI,DEF,PHY,Acceleration,Sprint Speed,Positioning,Finishing,Shot Power,Long Shots,Volleys,Penalties,Vision,Crossing,Free Kick Accuracy,Short Passing,Long Passing,Curve,Dribbling,Agility,Balance,Reactions,Ball Control,Composure,Interceptions,Heading Accuracy,Def Awareness,Standing Tackle,Sliding Tackle,Jumping,Stamina,Strength,Aggression,GK Diving,GK Handling,GK Kicking,GK Positioning,GK Reflexes
0,1,Kylian Mbappé,91,ST,LW,France,LALIGA EA SPORTS,Real Madrid,27,182,75,Right,4,5,"Finesse Shot, Rapid, Flair, Trivela, Acrobatic...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,97,90,80,92,36,78,97,97,93,94,90,83,84,84,83,78,69,86,71,80,93,93,82,93,92,88,38,73,26,34,32,88,88,77,64,13,5,7,11,6
1,2,Rodri,91,CDM,CM,Spain,Premier League,Manchester City,29,191,82,Right,4,3,"Power Shot, Long Ball Pass, Bruiser, Press Pro...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,66,80,86,84,87,85,65,66,76,74,92,89,71,62,84,76,64,93,91,86,84,66,67,93,90,94,84,81,92,87,82,83,91,83,85,10,10,7,14,8
2,4,Erling Haaland,91,ST,,Norway,Premier League,Manchester City,25,195,94,Left,3,3,"Power Shot, Power Header, Bruiser, Press Prove...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,88,92,70,81,45,88,80,94,96,96,94,83,90,90,74,58,62,77,66,77,79,77,69,94,83,87,43,83,38,47,29,92,76,93,88,7,14,13,11,7
3,5,Jude Bellingham,90,CAM,CM,England,LALIGA EA SPORTS,Real Madrid,22,186,75,Right,4,4,"Intercept, Slide Tackle, Technical, Flair, Rel...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,80,87,83,88,78,83,81,80,91,90,85,86,77,74,90,66,68,89,89,73,89,82,79,91,89,87,82,75,77,79,77,84,93,77,85,14,11,10,5,8
4,7,Vini Jr.,90,LW,"ST, LM",Brazil,LALIGA EA SPORTS,Real Madrid,25,176,73,Right,4,5,"Finesse Shot, Chip Shot, Rapid, Flair, First T...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,95,84,81,91,29,69,95,95,87,89,81,83,73,71,85,81,62,83,77,79,93,94,84,86,90,83,26,50,32,25,18,74,84,65,58,5,7,7,7,10


In [4]:
# Women's football (~1.5k players, ~16 requests)
female_players = fetch_players(gender=1)
female_players.head()


  gender=1: 1565/1565 done


,Rank,Name,OVR,Position,Alternative positions,Nation,League,Team,Age,Height,Weight,Preferred foot,Weak foot,Skill moves,play style,Gender,url,PAC,SHO,PAS,DRI,DEF,PHY,Acceleration,Sprint Speed,Positioning,Finishing,Shot Power,Long Shots,Volleys,Penalties,Vision,Crossing,Free Kick Accuracy,Short Passing,Long Passing,Curve,Dribbling,Agility,Balance,Reactions,Ball Control,Composure,Interceptions,Heading Accuracy,Def Awareness,Standing Tackle,Sliding Tackle,Jumping,Stamina,Strength,Aggression,GK Diving,GK Handling,GK Kicking,GK Positioning,GK Reflexes
0,3,Aitana Bonmatí,91,CM,CAM,Spain,Liga F,FC Barcelona,28,162,53,Right,5,4,"Finesse Shot, Incisive Pass, Pinged Pass, Tiki...",Women's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,81,86,86,91,77,75,82,80,91,91,79,89,70,70,90,74,77,91,90,78,92,93,86,91,91,85,88,54,75,81,67,75,82,75,64,9,15,11,12,14
1,6,Alexia Putellas,90,CM,CAM,Spain,Liga F,FC Barcelona,32,173,69,Left,5,5,"Finesse Shot, Incisive Pass, Pinged Pass, Tiki...",Women's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,82,89,90,91,72,78,81,82,91,91,86,89,90,89,91,88,89,91,90,89,90,90,89,90,92,92,78,74,60,81,64,84,85,78,70,15,17,11,15,10
2,8,Caroline Graham Hansen,90,RW,RM,Norway,Liga F,FC Barcelona,31,175,59,Right,5,5,"Finesse Shot, Technical, Flair, First Touch, Q...",Women's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,89,87,88,90,47,76,90,88,87,86,88,89,83,79,89,90,77,89,84,92,93,94,75,83,90,84,35,71,45,50,46,84,87,74,66,17,11,15,13,9
3,10,Sam Kerr,90,ST,,Australia,Barclays WSL,Chelsea,32,168,66,Right,4,4,"Power Header, Rapid, Quick Step, Acrobatic, Ae...",Women's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,85,88,74,90,42,83,86,84,92,92,85,85,92,71,79,70,68,78,67,76,90,90,82,89,91,91,24,93,44,39,30,89,87,86,70,7,12,8,16,13
4,14,Sophia Smith,89,ST,RW,United States,NWSL,Portland Thorns,25,164,58,Right,5,4,"Finesse Shot, Technical, Flair, Trickster, Qui...",Women's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,93,86,79,89,45,84,92,93,89,89,86,82,84,78,74,73,72,88,75,74,89,85,86,89,91,80,43,79,41,42,39,92,93,85,68,12,13,13,13,6


In [5]:
# Stats already arrive as integers, but coerce defensively (nullable Int64
# tolerates the rare missing value, e.g. GK stats on outfield players).
integer_columns = ['OVR', 'PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY',
       'Acceleration', 'Sprint Speed', 'Positioning', 'Finishing',
       'Shot Power', 'Long Shots', 'Volleys', 'Penalties', 'Vision', 'Crossing',
       'Free Kick Accuracy', 'Short Passing', 'Long Passing',
       'Curve', 'Dribbling', 'Agility', 'Balance', 'Reactions', 'Ball Control',
       'Composure', 'Interceptions', 'Heading Accuracy', 'Def Awareness',
       'Standing Tackle', 'Sliding Tackle', 'Jumping', 'Stamina', 'Strength',
       'Aggression', 'GK Diving', 'GK Handling', 'GK Kicking',
       'GK Positioning', 'GK Reflexes']

for df in (male_players, female_players):
    for col in integer_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')


In [6]:
all_players = pd.concat([male_players, female_players], ignore_index=True)
print('male:', male_players.shape, 'female:', female_players.shape, 'all:', all_players.shape)
print(all_players.columns.tolist())
all_players.head()


male: (15905, 57) female: (1565, 57) all: (17470, 57)
['Rank', 'Name', 'OVR', 'Position', 'Alternative positions', 'Nation', 'League', 'Team', 'Age', 'Height', 'Weight', 'Preferred foot', 'Weak foot', 'Skill moves', 'play style', 'Gender', 'url', 'PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY', 'Acceleration', 'Sprint Speed', 'Positioning', 'Finishing', 'Shot Power', 'Long Shots', 'Volleys', 'Penalties', 'Vision', 'Crossing', 'Free Kick Accuracy', 'Short Passing', 'Long Passing', 'Curve', 'Dribbling', 'Agility', 'Balance', 'Reactions', 'Ball Control', 'Composure', 'Interceptions', 'Heading Accuracy', 'Def Awareness', 'Standing Tackle', 'Sliding Tackle', 'Jumping', 'Stamina', 'Strength', 'Aggression', 'GK Diving', 'GK Handling', 'GK Kicking', 'GK Positioning', 'GK Reflexes']


,Rank,Name,OVR,Position,Alternative positions,Nation,League,Team,Age,Height,Weight,Preferred foot,Weak foot,Skill moves,play style,Gender,url,PAC,SHO,PAS,DRI,DEF,PHY,Acceleration,Sprint Speed,Positioning,Finishing,Shot Power,Long Shots,Volleys,Penalties,Vision,Crossing,Free Kick Accuracy,Short Passing,Long Passing,Curve,Dribbling,Agility,Balance,Reactions,Ball Control,Composure,Interceptions,Heading Accuracy,Def Awareness,Standing Tackle,Sliding Tackle,Jumping,Stamina,Strength,Aggression,GK Diving,GK Handling,GK Kicking,GK Positioning,GK Reflexes
0,1,Kylian Mbappé,91,ST,LW,France,LALIGA EA SPORTS,Real Madrid,27,182,75,Right,4,5,"Finesse Shot, Rapid, Flair, Trivela, Acrobatic...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,97,90,80,92,36,78,97,97,93,94,90,83,84,84,83,78,69,86,71,80,93,93,82,93,92,88,38,73,26,34,32,88,88,77,64,13,5,7,11,6
1,2,Rodri,91,CDM,CM,Spain,Premier League,Manchester City,29,191,82,Right,4,3,"Power Shot, Long Ball Pass, Bruiser, Press Pro...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,66,80,86,84,87,85,65,66,76,74,92,89,71,62,84,76,64,93,91,86,84,66,67,93,90,94,84,81,92,87,82,83,91,83,85,10,10,7,14,8
2,4,Erling Haaland,91,ST,,Norway,Premier League,Manchester City,25,195,94,Left,3,3,"Power Shot, Power Header, Bruiser, Press Prove...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,88,92,70,81,45,88,80,94,96,96,94,83,90,90,74,58,62,77,66,77,79,77,69,94,83,87,43,83,38,47,29,92,76,93,88,7,14,13,11,7
3,5,Jude Bellingham,90,CAM,CM,England,LALIGA EA SPORTS,Real Madrid,22,186,75,Right,4,4,"Intercept, Slide Tackle, Technical, Flair, Rel...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,80,87,83,88,78,83,81,80,91,90,85,86,77,74,90,66,68,89,89,73,89,82,79,91,89,87,82,75,77,79,77,84,93,77,85,14,11,10,5,8
4,7,Vini Jr.,90,LW,"ST, LM",Brazil,LALIGA EA SPORTS,Real Madrid,25,176,73,Right,4,5,"Finesse Shot, Chip Shot, Rapid, Flair, First T...",Men's Football,https://www.ea.com/games/ea-sports-fc/ratings/...,95,84,81,91,29,69,95,95,87,89,81,83,73,71,85,81,62,83,77,79,93,94,84,86,90,83,26,50,32,25,18,74,84,65,58,5,7,7,7,10


In [7]:
# Saved next to this notebook (backend/scripts/). index=False keeps the CSV clean.
male_players.to_csv('male_players.csv', index=False)
female_players.to_csv('female_players.csv', index=False)
all_players.to_csv('all_players.csv', index=False)
print('Saved male_players.csv, female_players.csv, all_players.csv')


Saved male_players.csv, female_players.csv, all_players.csv
